# Greyscale ablation — per-biomarker (categorical) concordance

**Isolated experiment, branch `greyscale-experiment`.** NEW analysis only; no Phase 1–4 module,
Prompt 3's `greyscale_concordance.py`, config, prediction, or report is modified. All outputs
under `greyscale_experiment/`. CPU fine.

Prompt 3 found strong **continuous** concordance (harmonised grade↔CST ρ = 0.332, grade↔burden
ρ = 0.486). This runs the stronger **categorical** check: can the harmonised predicted grade
detect individual OLIVES biomarkers (per-biomarker AUC on the ~192 tier-2 samples), harmonised
vs original, with stratified bootstrap CIs — plus DME composites, the grade≥2 operating point,
and patient-level multi-visit consistency. Per-biomarker labels come from the raw
`olives_fundus.pt` [N,16] biomarkers tensor, index-aligned to the CSV and masked to tier-2.

In [ ]:
# Setup: mount Drive, restore the repo on the greyscale-experiment branch, cd in.
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_DIR = '/content/dr-dissertation'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/savita10/dr-dissertation.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin && git checkout greyscale-experiment && git pull

## Run STEP 1 introspection + the categorical concordance analysis
The module prints STEP 1 (predictions filename + tier breakdown; OLIVES tensor keys, biomarkers
shape, has_biomarkers count; per-biomarker tier-2 positive rates vs the expected table; confirms
the Prompt 3 concordance JSON and Phase 4b JSON are readable), then runs per-biomarker AUC,
composites, grade≥2 operating points, and patient-level agreement — writing the JSON, report,
and 3 figures.

In [ ]:
!python -m src.analysis.greyscale_categorical_concordance --config configs/greyscale_eval.yaml

## Display the report + figures

In [ ]:
from IPython.display import Markdown, Image, display
from pathlib import Path

root = Path('/content/drive/MyDrive/dissertation/greyscale_experiment')
display(Markdown((root / 'greyscale_categorical_concordance_report.md').read_text()))

figs = [
    'greyscale_biomarker_auc_barchart.png',
    'greyscale_composite_roc.png',
    'greyscale_biomarker_confusion.png',
]
for name in figs:
    p = root / 'figures' / name
    if p.exists():
        display(Image(str(p)))

## Done
Verdict (categorical-recovered / aggregate-only / categorical-fails) is at the top of the report,
with the four primary biomarker AUCs (harmonised vs original), the two composite AUCs, and the
clinically-interpretable claim: screening at grade≥2 detects X% of DME-composite-positive eyes at
Y% specificity. Per-biomarker CIs are wide (~192 tier-2 samples) — a positive is strong indirect
evidence, not proof. Commit/push from PowerShell on `greyscale-experiment`.